# 🏆 Hướng Dẫn Huấn Luyện ViSEC Pitch-Fusion Baseline Trên Kaggle GPU

Notebook này thiết lập môi trường hoàn chỉnh để chạy mô hình **ViSEC Pitch-Fusion Baseline (ICASSP 2024)** trên nền tảng **Kaggle GPU** (P100 hoặc T4 x2) với tập dữ liệu được phân chia đồng bộ **80/10/10 (Seed 42)**.

---

### ⚙️ Cài đặt cấu hình Kaggle bắt buộc trước khi chạy:
Ở thanh công cụ bên phải (Notebook Settings):
1. **Accelerator**: Chọn **GPU P100** hoặc **GPU T4 x 2**.
2. **Internet**: Bật **Internet ON** (Bắt buộc để tải dataset Hugging Face và weights Wav2Vec 2.0).
3. **Persistence**: Variables and Files (Tùy chọn).

### Bước 1: Kiểm tra cấu hình GPU Kaggle

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name:        {torch.cuda.get_device_name(0)}")

### Bước 2: Chuẩn bị mã nguồn dự án trên Kaggle
Tải mã nguồn mới nhất chứa module `visec_baseline` về thư mục làm việc `/kaggle/working`.

In [ ]:
import os
import shutil

# Đảm bảo đứng ở thư mục gốc /kaggle/working
%cd /kaggle/working

# Clone mã nguồn từ GitHub (đã cập nhật module visec_baseline)
if os.path.exists('MaxMViT-MLP-SER'):
    shutil.rmtree('MaxMViT-MLP-SER')

!git clone https://github.com/Huu2412/MaxMViT-MLP-SER.git
%cd /kaggle/working/MaxMViT-MLP-SER

### Bước 3: Cài đặt các thư viện phụ thuộc

In [ ]:
!pip install -q transformers datasets torchaudio accelerate soundfile librosa scikit-learn seaborn matplotlib pyyaml

### Bước 4: Bắt đầu Huấn Luyện Mô Hình ViSEC Pitch-Fusion (ICASSP 2024)

Cấu hình đã tối ưu chống tràn bộ nhớ CUDA VRAM:
- **Batch Size: 2** + **Gradient Accumulation: 4** $\rightarrow$ Kích thước batch tương đương: 8 (giảm 50% đỉnh bộ nhớ).
- **Gradient Checkpointing**: Giảm ~60% bộ nhớ activations của Wav2Vec 2.0.
- **Max Duration: 8.0s**: Giới hạn độ dài các file âm thanh ngoại lai dài bất thường để tránh OOM.
- **Expandable Segments**: Chống phân mảnh bộ nhớ PyTorch CUDA.

In [ ]:
# Tối ưu bộ nhớ CUDA: chống phân mảnh bộ nhớ & thiết lập GPU
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

!python run_visec_baseline.py \
    --mode train_pitch \
    --batch_size 2 \
    --grad_accum 4 \
    --max_duration 8.0 \
    --gradient_checkpointing \
    --epochs 30 \
    --lr 1.5e-5 \
    --num_proc 4 \
    --seed 42 \
    --fp16 \
    --output_dir /kaggle/working/checkpoints/visec_pitch_baseline

### (Tùy chọn) Huấn luyện Baseline Wav2Vec 2.0 gốc (Không có Pitch)
Bỏ chú thích (uncomment) nếu bạn muốn đo lường hiệu năng của Wav2Vec 2.0 chuẩn để so sánh đối chứng.

In [ ]:
# !python run_visec_baseline.py \
#     --mode train_no_joint \
#     --batch_size 2 \
#     --grad_accum 4 \
#     --epochs 30 \
#     --fp16 \
#     --output_dir /kaggle/working/checkpoints/visec_no_joint_baseline

### Bước 5: Xem Kết Quả Chi Tiết & Confusion Matrix Trên Tập Test

In [ ]:
from IPython.display import Image, display
import json

results_json = '/kaggle/working/checkpoints/visec_pitch_baseline/test_results.json'
cm_image = '/kaggle/working/checkpoints/visec_pitch_baseline/confusion_matrix.png'

if os.path.exists(results_json):
    with open(results_json, 'r', encoding='utf-8') as f:
        metrics = json.load(f)
        
    print("=" * 60)
    print("  KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP (528 MẪU)")
    print("=" * 60)
    print(f"  - Macro F1-score:      {metrics['macro_f1']*100:.2f}%")
    print(f"  - Balanced Accuracy:   {metrics['balanced_accuracy']*100:.2f}%")
    print(f"  - Overall Accuracy:    {metrics['accuracy']*100:.2f}%")
    print("\nClassification Report:")
    print(metrics['classification_report'])
    
    if os.path.exists(cm_image):
        print("\nBiểu đồ Confusion Matrix:")
        display(Image(cm_image))
else:
    print("Chưa tìm thấy file kết quả. Hãy đảm bảo quá trình huấn luyện ở Bước 4 đã hoàn tất.")

### Bước 6: Đóng gói Checkpoint & Kết quả để tải về máy
Tạo file zip chứa mô hình tốt nhất (`best_model`), kết quả test (`test_results.json`, `classification_report.txt`) và hình ảnh ma trận nhầm lẫn để bạn tải về từ tab **Output** của Kaggle.

In [ ]:
%cd /kaggle/working
!zip -r visec_pitch_baseline_results.zip checkpoints/visec_pitch_baseline/
print("\n[XONG] File 'visec_pitch_baseline_results.zip' đã sẵn sàng trong thư mục /kaggle/working!")
print("Bạn có thể tải về trực tiếp từ panel Output bên phải của Kaggle.")